In [17]:
import requests


# start a spark session and create a spark context
from pyspark.sql import SparkSession
spark = SparkSession.builder \
    .appName("nyc_taxi") \
    .getOrCreate()

sc = spark.sparkContext

In [18]:
# set dl url for January 2019 trip data
download_url = 'https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2019-01.parquet'

# get the data
response = requests.get(download_url)

# check that response was good and save the data
jan_2019_trip_data = "yellow_tripdata_2019-01.parquet"
if response.status_code == 200:
    # Changed 'response' to 'data' here
    with open(jan_2019_trip_data, "wb") as f:
        f.write(response.content)

In [19]:
# create the dataframe
df_trips = spark.read.parquet(jan_2019_trip_data)   

In [20]:
# Show the dataframe
df_trips.show()

+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+
|VendorID|tpep_pickup_datetime|tpep_dropoff_datetime|passenger_count|trip_distance|RatecodeID|store_and_fwd_flag|PULocationID|DOLocationID|payment_type|fare_amount|extra|mta_tax|tip_amount|tolls_amount|improvement_surcharge|total_amount|congestion_surcharge|airport_fee|
+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+
|       1| 2019-01-01 00:46:40|  2019-01-01 00:53:20|            1.0|          1.5|       1.0|                 N|         151|         239|           1|        7.0|  0.5|    0.5|      1.6

In [21]:
from pyspark.sql.functions import monotonically_increasing_id

df_trips = df_trips.withColumn("trip_id", monotonically_increasing_id())
df_trips.show(5)

+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+-----------+
|VendorID|tpep_pickup_datetime|tpep_dropoff_datetime|passenger_count|trip_distance|RatecodeID|store_and_fwd_flag|PULocationID|DOLocationID|payment_type|fare_amount|extra|mta_tax|tip_amount|tolls_amount|improvement_surcharge|total_amount|congestion_surcharge|airport_fee|    trip_id|
+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+-----------+
|       1| 2019-01-01 00:46:40|  2019-01-01 00:53:20|            1.0|          1.5|       1.0|                 N|         151|         239|           1

In [22]:
from pyspark.sql.functions import max, avg


df_trips.select(max("passenger_count")).show()

df_trips.select(avg("passenger_count")).show()

+--------------------+
|max(passenger_count)|
+--------------------+
|                 9.0|
+--------------------+

+--------------------+
|avg(passenger_count)|
+--------------------+
|  1.5670317144945614|
+--------------------+



In [23]:
from pyspark.sql.functions import col, unix_timestamp, to_date, hour, dayofweek, count

df_trips = df_trips.withColumn(
    "trip_duration_mins",
    (unix_timestamp(col("tpep_dropoff_datetime")) - unix_timestamp(col("tpep_pickup_datetime"))) / 60
)

In [24]:
print("--- Trajets par Distance ---")
df_trips.selectExpr("MAX(trip_distance) as max_distance", "MIN(trip_distance) as min_distance").show()

print("--- Trajets par Temps (en minutes) ---")
df_trips.selectExpr("MAX(trip_duration_mins) as max_duration_mins", "MIN(trip_duration_mins) as min_duration_mins").show()

--- Trajets par Distance ---
+------------+------------+
|max_distance|min_distance|
+------------+------------+
|       831.8|         0.0|
+------------+------------+

--- Trajets par Temps (en minutes) ---
+-----------------+-----------------+
|max_duration_mins|min_duration_mins|
+-----------------+-----------------+
|43648.01666666667|         -84280.5|
+-----------------+-----------------+



In [25]:
daily_trips = df_trips.withColumn("pickup_date", to_date(col("tpep_pickup_datetime"))) \
                      .groupBy("pickup_date") \
                      .agg(count("*").alias("total_trips"))

print("Jour le plus chargé :")
daily_trips.orderBy(col("total_trips").desc()).show(1)

print("Jour le plus calme :")
daily_trips.orderBy(col("total_trips").asc()).show(1)

Jour le plus chargé :
+-----------+-----------+
|pickup_date|total_trips|
+-----------+-----------+
| 2019-01-25|     292499|
+-----------+-----------+
only showing top 1 row
Jour le plus calme :
+-----------+-----------+
|pickup_date|total_trips|
+-----------+-----------+
| 2019-05-20|          1|
+-----------+-----------+
only showing top 1 row


In [26]:
hourly_trips = df_trips.withColumn("pickup_hour", hour(col("tpep_pickup_datetime"))) \
                       .groupBy("pickup_hour") \
                       .agg(count("*").alias("total_trips"))

print("Heure la plus chargée :")
hourly_trips.orderBy(col("total_trips").desc()).show(1)

print("Heure la plus calme :")
hourly_trips.orderBy(col("total_trips").asc()).show(1)

Heure la plus chargée :
+-----------+-----------+
|pickup_hour|total_trips|
+-----------+-----------+
|         18|     515390|
+-----------+-----------+
only showing top 1 row
Heure la plus calme :
+-----------+-----------+
|pickup_hour|total_trips|
+-----------+-----------+
|          4|      61424|
+-----------+-----------+
only showing top 1 row


In [27]:
day_of_week_trips = df_trips.withColumn("day_of_week", dayofweek(col("tpep_pickup_datetime"))) \
                            .groupBy("day_of_week") \
                            .agg(count("*").alias("total_trips"))

print("Tendance par jour de la semaine (1=Dimanche, 7=Samedi) :")
day_of_week_trips.orderBy(col("total_trips").desc()).show()

Tendance par jour de la semaine (1=Dimanche, 7=Samedi) :
+-----------+-----------+
|day_of_week|total_trips|
+-----------+-----------+
|          5|    1357043|
|          4|    1265264|
|          3|    1209084|
|          6|    1087215|
|          7|    1009985|
|          2|     908121|
|          1|     859905|
+-----------+-----------+



In [28]:

print("Corrélation Distance/ Pourboire :", df_trips.stat.corr("trip_distance", "tip_amount"))
print("Corrélation Passager/ Pourboire :", df_trips.stat.corr("passenger_count", "tip_amount"))

print("Trajet avec la chrge 'extra' maximale :")
df_trips.orderBy(col("extra").desc()).select("trip_id", "tpep_pickup_datetime", "extra", "total_amount").show(1)

Corrélation Distance/ Pourboire : 0.5269200663652668
Corrélation Passager/ Pourboire : 0.004431051585116288
Trajet avec la chrge 'extra' maximale :
+-----------+--------------------+------+------------+
|    trip_id|tpep_pickup_datetime| extra|total_amount|
+-----------+--------------------+------+------------+
|42954996443| 2019-01-23 08:58:09|535.38|   356214.78|
+-----------+--------------------+------+------------+
only showing top 1 row


### Analyse des valeurs aberrantes (Outliers)

L'exploration initiale du jeu de données révele plusieurs incohérences majeures qui nécessitent un nettoyage de la donnée avant toute modélisation ou analyse métier avancée. Voici les anomalies identifiées :

* **Données hors période cible :** Le jeux de données est documente comme contenant exclusivement les trajets de janvier 2019 Cependant, l'affichage des premières lignes du DataFrame et l'agrégation par date révèlent la présence de course enregistrées en novembre 2018, décembre 2018, et même en mai 2019.
* **Anomalies de distances (GPS / Compteur) :** La distance maximale enregistrée est de 831,8 miles. Une course de taxi de plus de 1300 kilomètres au sein de New York est irréaliste et traduit une erreur matérielle du capteur. À l'inverse, de nombreuses courses affichent une distance de 0,0 mile, ce qui correspond  probablement, à des courses annulées immédiatement après le déclechement du compteur ou à un échec de la liaison GPS.
* **Anomalies de durées (Horodatage) :** Le calcul des durees de trajet met en évidence des valeurs negatives allant jusqu'à -84 280,5 minutes[cite: 20], indiquant une défaillance de l'horloge système ou une inversion informatique entre lheure de prise en charge et l'heure de dépose. La durée maximale de 43 648 minutes correspond à un trajet continu d'environ un mois, ce qui est physiqement impossible pour un taxi[cite: 20].

In [29]:
# Téléchargement du Taxi Zone Lookup
zone_url = 'https://d37ci6vzurychx.cloudfront.net/misc/taxi+_zone_lookup.csv'
response_zone = requests.get(zone_url)
with open("taxi_zone_lookup.csv", "wb") as f:
    f.write(response_zone.content)

# Chargement du CSV dans un DataFrame
df_zones = spark.read.format("csv").option("header", "true").load("taxi_zone_lookup.csv")

# Jointure pour les lieux de prise en charge (Pickups)
df_trips_zones = df_trips.join(df_zones, df_trips.PULocationID == df_zones.LocationID, "left") \
                         .withColumnRenamed("Borough", "Pickup_Borough") \
                         .withColumnRenamed("Zone", "Pickup_Zone") \
                         .drop("LocationID", "service_zone")

# Jointure pour les lieux de dépose (Dropoffs)
df_trips_zones = df_trips_zones.join(df_zones, df_trips_zones.DOLocationID == df_zones.LocationID, "left") \
                               .withColumnRenamed("Borough", "Dropoff_Borough") \
                               .withColumnRenamed("Zone", "Dropoff_Zone") \
                               .drop("LocationID", "service_zone")

In [30]:
print("--- Prises en charge par arrondissement ---")
df_trips_zones.groupBy("Pickup_Borough").agg(count("*").alias("total_pickups")) \
              .orderBy(col("total_pickups").desc()).show()

print("--- Déposes par arrondissement ---")
df_trips_zones.groupBy("Dropoff_Borough").agg(count("*").alias("total_dropoffs")) \
              .orderBy(col("total_dropoffs").desc()).show()

--- Prises en charge par arrondissement ---
+--------------+-------------+
|Pickup_Borough|total_pickups|
+--------------+-------------+
|     Manhattan|      6950965|
|        Queens|       471173|
|       Unknown|       159815|
|      Brooklyn|        91905|
|         Bronx|        18062|
|           N/A|         3890|
|           EWR|          446|
| Staten Island|          361|
+--------------+-------------+

--- Déposes par arrondissement ---
+---------------+--------------+
|Dropoff_Borough|total_dropoffs|
+---------------+--------------+
|      Manhattan|       6817355|
|         Queens|        340972|
|       Brooklyn|        301105|
|        Unknown|        149097|
|          Bronx|         58085|
|            N/A|         16904|
|            EWR|         10914|
|  Staten Island|          2185|
+---------------+--------------+



In [31]:
print("--- Moyennes par arrondissement de départ ---")
df_trips_zones.groupBy("Pickup_Borough") \
              .agg(avg("trip_distance").alias("avg_distance"),
                   avg("fare_amount").alias("avg_fare")) \
              .orderBy(col("avg_fare").desc()).show()

--- Moyennes par arrondissement de départ ---
+--------------+------------------+------------------+
|Pickup_Borough|      avg_distance|          avg_fare|
+--------------+------------------+------------------+
|           EWR| 2.641098654708519| 76.24024663677126|
|           N/A| 3.193850899742941|  59.5731593830335|
| Staten Island|12.503601108033246|45.289861495844896|
|        Queens|11.283218499361993| 35.14462651722029|
|         Bronx| 7.233194552098303| 26.26890543682963|
|      Brooklyn| 4.787677275447492|18.649132800172286|
|       Unknown| 2.415464130400774|14.944423051653523|
|     Manhattan|2.2286693358402596|10.792468572351568|
+--------------+------------------+------------------+



In [33]:
from pyspark.sql.functions import hour, col, count

print("--- Heures les plus chargées par Borough ---")
df_trips_zones.withColumn("pickup_hour", hour(col("tpep_pickup_datetime"))) \
              .groupBy("Pickup_Borough", "pickup_hour") \
              .agg(count("*").alias("total")) \
              .orderBy("Pickup_Borough", col("total").desc()) \
              .show(20)

--- Heures les plus chargées par Borough ---
+--------------+-----------+-----+
|Pickup_Borough|pickup_hour|total|
+--------------+-----------+-----+
|         Bronx|          7| 1803|
|         Bronx|          8| 1445|
|         Bronx|          6| 1301|
|         Bronx|          9| 1158|
|         Bronx|         10| 1079|
|         Bronx|         14| 1014|
|         Bronx|         12|  956|
|         Bronx|         13|  918|
|         Bronx|         15|  897|
|         Bronx|         11|  885|
|         Bronx|         17|  812|
|         Bronx|         16|  756|
|         Bronx|         18|  741|
|         Bronx|          5|  736|
|         Bronx|         19|  519|
|         Bronx|         20|  440|
|         Bronx|         23|  410|
|         Bronx|         21|  408|
|         Bronx|          4|  403|
|         Bronx|         22|  353|
+--------------+-----------+-----+
only showing top 20 rows


In [34]:
# Trajet le plus cher et son arrondissement de départ
print("--- Tarif le plus élevé ---")
df_trips_zones.orderBy(col("fare_amount").desc()) \
              .select("Pickup_Borough", "fare_amount", "trip_distance").show(1)

# Trajet le moins cher (souvent négatif)
print("--- Tarif le plus bas ---")
df_trips_zones.orderBy(col("fare_amount").asc()) \
              .select("Pickup_Borough", "fare_amount", "trip_distance").show(1)

--- Tarif le plus élevé ---
+--------------+-----------+-------------+
|Pickup_Borough|fare_amount|trip_distance|
+--------------+-----------+-------------+
|     Manhattan|  623259.86|          2.4|
+--------------+-----------+-------------+
only showing top 1 row
--- Tarif le plus bas ---
+--------------+-----------+-------------+
|Pickup_Borough|fare_amount|trip_distance|
+--------------+-----------+-------------+
|        Queens|     -362.0|          0.0|
+--------------+-----------+-------------+
only showing top 1 row


In [35]:
# Téléchargement des données de Janvier 2025
url_2025 = 'https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2025-01.parquet'
jan_2025_trip_data = "yellow_tripdata_2025-01.parquet"

response = requests.get(url_2025)
if response.status_code == 200:
    with open(jan_2025_trip_data, "wb") as f:
        f.write(response.content)

# Chargement du DataFrame 2025
df_2025 = spark.read.parquet(jan_2025_trip_data)

# Calcul des métriques globales pour comparer avec 2019
print("--- Métriques Janvier 2025 ---")
df_2025.selectExpr(
    "COUNT(*) as total_trips", 
    "AVG(trip_distance) as avg_distance", 
    "AVG(fare_amount) as avg_fare"
).show()

--- Métriques Janvier 2025 ---
+-----------+-----------------+-----------------+
|total_trips|     avg_distance|         avg_fare|
+-----------+-----------------+-----------------+
|    3475226|5.855126178843539|17.08180276045484|
+-----------+-----------------+-----------------+



In [ ]:
#partie3

df_trips.createOrReplaceTempView("trips_2019")
df_zones.createOrReplaceTempView("taxi_zones")

print("--- SQL 1 : Jourle plus chargé ---")
spark.sql("""
    SELECT DATE(tpep_pickup_datetime) as pickup_date, COUNT(*) as total_trips
    FROM trips_2019
    GROUP BY DATE(tpep_pickup_datetime)
    ORDER BY total_trips DESC
    LIMIT 1
""").show()

print("--- SQL 2 : Tarif le plus éleve ---")
spark.sql("""
    SELECT trip_id, fare_amount, trip_distance 
    FROM trips_2019 
    ORDER BY fare_amount DESC 
    LIMIT 1
""").show()

print("--- SQL 3 : Total de trajet par arrondissement de depart (JOIN) ---")
spark.sql("""
    SELECT z.Borough as Pickup_Borough, COUNT(t.trip_id) as total_pickups
    FROM trips_2019 t
    LEFT JOIN taxi_zones z ON t.PULocationID = z.LocationID
    GROUP BY z.Borough
    ORDER BY total_pickups DESC
""").show()

--- SQL 1 : Jour le plus chargé ---
+-----------+-----------+
|pickup_date|total_trips|
+-----------+-----------+
| 2019-01-25|     292499|
+-----------+-----------+

--- SQL 2 : Tarif le plus élevé ---
+-----------+-----------+-------------+
|    trip_id|fare_amount|trip_distance|
+-----------+-----------+-------------+
|42952172615|  623259.86|          2.4|
+-----------+-----------+-------------+

--- SQL 3 : Total de trajets par arrondissement de départ (avec JOIN) ---
+--------------+-------------+
|Pickup_Borough|total_pickups|
+--------------+-------------+
|     Manhattan|      6950965|
|        Queens|       471173|
|       Unknown|       159815|
|      Brooklyn|        91905|
|         Bronx|        18062|
|           N/A|         3890|
|           EWR|          446|
| Staten Island|          361|
+--------------+-------------+

